# 04 — Impact du Top-K

## Objectif

Étudier l'impact du nombre de chunks récupérés (Top-K) sur la qualité
des réponses.

**Question de recherche :** Quel Top-K offre le meilleur compromis
entre pertinence et temps de réponse ?

**Paramètres fixes :**
- LLM : Gemini 2.5 Flash
- Embedding : BAAI/bge-m3
- 3 questions de test

---
**Pourquoi cette comparaison est importante :**
Top-K contrôle la quantité d'information donnée au LLM. Trop peu = info
manquante, trop = bruit et temps excessif.

## 1. Imports

In [ ]:
import sys
from pathlib import Path
import time

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "src").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

from config import LLMConfig, RetrievalConfig, EmbeddingConfig
from retriever import retrieve_documents
from llm_chain import generate_answer
from evaluation_judge import create_judge
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

print("✅ Imports réussis")

## 2. Paramètres fixes

In [ ]:
llm_cfg = LLMConfig(
    provider="openrouter", model="google/gemini-2.5-flash",
    temperature=0.2, num_predict=512, request_timeout=120,
)
emb_cfg = EmbeddingConfig(provider="huggingface", model="BAAI/bge-m3", device="cpu")

print(f"🤖 LLM       : {llm_cfg.provider}:{llm_cfg.model}")
print(f"🔤 Embedding : {emb_cfg.provider}:{emb_cfg.model}")

## 3. Valeurs de Top-K à tester

On teste k = 1, 3, 5, 7, 10 pour observer la tendance.

In [ ]:
TOP_K_VALUES = [1, 3, 5, 7, 10]
print(f"📋 Top-K testés : {TOP_K_VALUES}")

## 4. Questions de test

In [ ]:
QUESTIONS = [
    ("Q01", "Quelle est la note minimale pour valider un module ?",
     "L'étudiant doit obtenir une note finale >= 5,5/10"),
    ("Q02", "Peut-on demander une prolongation du mémoire ?",
     "Oui, avec une demande écrite"),
    ("Q03", "Quel est le montant des frais de prolongation ?",
     "1 000 000 VND par mois"),
]

print(f"📚 {len(QUESTIONS)} questions")

## 5. Juge DeepEval

In [ ]:
judge = create_judge(provider="ollama", model="qwen2.5:3b")
print(f"⚖️  Juge : {judge.get_model_name()}")

## 6. Boucle d'évaluation

Pour chaque valeur de Top-K, on exécute le pipeline complet.

In [ ]:
results = []

for k in TOP_K_VALUES:
    print(f"\n{'='*60}")
    print(f"  🔄 Test : top_k = {k}")
    print(f"{'='*60}")

    ret_cfg = RetrievalConfig(top_k=k, max_distance=1.5)

    for qid, question, expected in QUESTIONS::
        try:
        docs, scores = retrieve_documents(question, ret_cfg, emb_cfg)

        start = time.time()
        answer = generate_answer(question, docs, llm_cfg)
        elapsed = time.time() - start

        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            expected_output=expected,
            retrieval_context=[d.page_content for d in docs],
        )

        faith = FaithfulnessMetric(threshold=0.75, model=judge, include_reason=True)
        relev = AnswerRelevancyMetric(threshold=0.75, model=judge, include_reason=True)

        try:
            faith.measure(test_case)
        except Exception:
            faith.score = 0.0
        try:
            relev.measure(test_case)
        except Exception:
            relev.score = 0.0

        results.append({
            "Top-K": k,
            "Question": qid,
            "Faithfulness": round(faith.score, 4),
            "AnswerRelevancy": round(relev.score, 4),
            "Temps(s)": round(elapsed, 2),
            "Chunks": len(docs),
        })

        except Exception as e:
            print(f"   [04] ❌ Erreur : {e}")
                    nt(f"   [{qid}] Faith={faith.score:.3f}  Relev={relev.score:.3f}  chunks={len(docs)}")

df = pd.DataFrame(results)
print(f"\n✅ Terminé : {len(df)} mesures")

## 7. Tableau des moyennes par Top-K

In [ ]:
summary = df.groupby("Top-K")[["Faithfulness", "AnswerRelevancy", "Temps(s)", "Chunks"]].mean().round(4)
summary.columns = ["Fidélité", "Pertinence", "Temps (s)", "Chunks"]
summary

## 8. Graphique d'évolution

Ce graphique montre comment les métriques évoluent avec Top-K.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(summary.index, summary["Fidélité"], marker="o", linewidth=2, label="Fidélité")
ax1.plot(summary.index, summary["Pertinence"], marker="s", linewidth=2, label="Pertinence")
ax1.set_xlabel("Top-K")
ax1.set_ylabel("Score")
ax1.set_title("Impact de Top-K sur les métriques", fontsize=14, fontweight="bold")
ax1.set_ylim(0, 1)
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Temps de réponse
fig2, ax2 = plt.subplots(figsize=(10, 3))
ax2.plot(summary.index, summary["Temps (s)"], marker="s", color="red", linewidth=2)
ax2.set_xlabel("Top-K")
ax2.set_ylabel("Temps (s)")
ax2.set_title("Temps de réponse", fontsize=14, fontweight="bold")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Export CSV

In [ ]:
out_dir = ROOT / "evaluation" / "results"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "04_compare_topk.csv", index=False)
summary.to_csv(out_dir / "04_compare_topk_summary.csv")
print(f"✅ Exporté dans {out_dir}/")

## 10. Analyse

**Lecture des résultats :**
- Si la **Fidélité** augmente avec k → le LLM a besoin de plus de contexte
- Si les scores stagnent après un certain k → valeur optimale atteinte
- Si la **Pertinence** baisse à haut k → le bruit nuit à la réponse
- Le temps de réponse augmente avec k (plus de tokens en entrée)

**Recommandation :** choisir la plus petite valeur de k qui maximise
les métriques, pour minimiser le temps et le bruit.